IT Incident Ticket Analysis

Identifying Drivers of Resolution Time in Enterprise IT Operations


0. Objective & Motivation

Objective:
The goal of this analysis is to examine IT incident ticket data to identify factors that contribute to longer resolution times and operational bottlenecks. Understanding these drivers can inform process improvements, resource allocation, and SLA management.

Why this matters:
In enterprise IT environments, prolonged incident resolution impacts employee productivity and operational reliability. Data-driven insights can help prioritize improvements and reduce mean time to resolution (MTTR).


1. Dataset Overview & Context

Dataset description:
This dataset represents an incident event log from an IT service management (ITSM) system similar to ServiceNow. Each row corresponds to an event or update associated with an incident rather than a single incident record.

Key characteristics:

Multiple rows per incident

State changes recorded over time

Timestamps reflect different stages of the incident lifecycle

Unit of analysis (raw data):
Event-level updates
Unit of analysis (after transformation):
One row per incident

2. Understanding the Event Log Structure

Before analysis, it is critical to understand how incidents evolve over time.

Important observations:

Incidents transition through multiple states (e.g., New → Resolved → Closed)

Priority, assignment group, and other attributes may change mid-incident

Naïvely treating each row as an independent ticket would lead to double-counting

Analytical implication:
The dataset must be transformed from an event-level log into an incident-level table.

3. Event-to-Incident Transformation

Purpose:
Collapse multiple event records into a single record per incident.

Aggregation strategy:

opened_at: earliest timestamp per incident

resolved_at: latest timestamp where state = “Resolved”

closed_at: final timestamp per incident

Priority: derived as initial, final, and maximum observed priority

Counts of:

reassignment events

reopen events

total updates per incident

Assumptions:

Resolution is defined as the first transition to a resolved state

Closure may occur after resolution for administrative reasons

In [175]:
#import necessary packages and load data

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv(r'C:\Users\Vishnu Ravi\python files\IT ticket data analysis project\incident_event_log.csv')
print(df.shape)
pd.set_option('display.max_columns', None)

#seeing that there are multiple rows per incident, we can see that the first 4 rows all have the same number, but show progression in incident_state
print(df.iloc[174])
print(df.index[df["resolved_at"] == "?"])

(119998, 36)
number                          INC0000093
incident_state                      Closed
active                               False
reassignment_count                       0
reopen_count                             0
sys_mod_count                            1
made_sla                              True
caller_id                      Caller 3545
opened_by                   Opened by  131
opened_at                  29/2/2016 08:56
sys_created_by                           ?
sys_created_at                           ?
sys_updated_by              Updated by 908
sys_updated_at              5/3/2016 10:00
contact_type                         Phone
location                       Location 93
category                       Category 61
subcategory                Subcategory 164
u_symptom                      Symptom 580
cmdb_ci                                  ?
impact                          2 - Medium
urgency                         2 - Medium
priority                      3 - Moderat

Several columns exhibit high missingness at the event level due to the process-driven nature of incident logging. Missing values do not necessarily indicate data quality issues, but rather reflect fields that are only populated at specific lifecycle stages (e.g., resolved_at, assigned_to).

Columns with extreme sparsity (>90%) and limited analytical value without external enrichment (e.g., problem_id, rfc, vendor) were removed. Core temporal fields such as opened_at, resolved_at, and closed_at were retained and handled conditionally during aggregation.

The way that resolved_at works is that after the resolution time is put in, it's backfilled for all other rows containing the same number, this can be seen by looking at rows that have tickets marked as 'New' or 'Active' which have resolution times even though they shouldn't. This also means that missing resolution times are true misses, and cannot be reconstructed by looking at the other rows that contain the same number. Imputation doesn't work for this data since it would be complete fabrication as there isn't enough information to predict a possible resolution time given the other factors


Data Quality and Assumptions

This dataset represents an event-level log of incident records, where each row is a snapshot of an incident at a given system update. Many fields are duplicated across rows and updated cumulatively over time. As a result, missing values often reflect process behavior rather than data corruption.

Several reference columns (problem_id, rfc, vendor, caused_by, cmdb_ci) were removed due to extreme sparsity (>90% missing) and limited analytical value without external enrichment.

Core temporal fields (opened_at, resolved_at, closed_at) were retained and treated conservatively. Inspection of event indices confirms that a subset of incidents (~2.3%) were closed without a recorded resolution timestamp. For these incidents, resolved_at is missing for all associated rows, indicating that the resolution time was never entered into the system and cannot be recovered. These incidents are retained and explicitly flagged rather than imputed or dropped.

Duration features are computed only when logically valid. Resolution-based metrics are left undefined (NaN) when no resolution timestamp exists. This preserves data integrity and allows downstream models to learn from missingness rather than obscuring it.

Cumulative operational counts (reassignments, reopens, updates) are aggregated using maximum values to avoid double-counting historical events.

In [ ]:
print(df.shape)


print(f"Number of missing values = {(df == '?').sum()}")


# the below 4 columns are empty >90% of the time, so we will drop them
df.drop(columns =['problem_id','rfc','vendor','caused_by','cmdb_ci'])

#Although sys_created_at has ~35% missing values at the event level, it is not used as a primary temporal anchor. The incident opening time is consistently captured by opened_at, which is complete. Missing sys_created_at values reflect logging sparsity rather than data corruption.


(119998, 36)
Number of missing values = number                          0
incident_state                  0
active                          0
reassignment_count              0
reopen_count                    0
sys_mod_count                   0
made_sla                        0
caller_id                      29
opened_by                    4835
opened_at                       0
sys_created_by              42354
sys_created_at              42354
sys_updated_by                  0
sys_updated_at                  0
contact_type                    0
location                       76
category                       64
subcategory                    97
u_symptom                   28271
cmdb_ci                    119562
impact                          0
urgency                         0
priority                        0
assignment_group            14213
assigned_to                 23030
knowledge                       0
u_priority_confirmation         0
notify                          0
problem_

,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,sys_created_by,sys_created_at,sys_updated_by,sys_updated_at,contact_type,location,category,subcategory,u_symptom,impact,urgency,priority,assignment_group,assigned_to,knowledge,u_priority_confirmation,notify,closed_code,resolved_by,resolved_at,closed_at
0,INC0000045,New,True,0,0,0,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 21,29/2/2016 01:23,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
1,INC0000045,Resolved,True,0,0,2,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 642,29/2/2016 08:53,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
2,INC0000045,Resolved,True,0,0,3,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 804,29/2/2016 11:29,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
3,INC0000045,Closed,False,0,0,4,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 908,5/3/2016 12:00,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
4,INC0000047,New,True,0,0,0,True,Caller 2403,Opened by 397,29/2/2016 04:40,Created by 171,29/2/2016 04:57,Updated by 746,29/2/2016 04:57,Phone,Location 165,Category 40,Subcategory 215,Symptom 471,2 - Medium,2 - Medium,3 - Moderate,Group 70,Resolver 89,True,False,Do Not Notify,code 5,Resolved by 81,1/3/2016 09:52,6/3/2016 10:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119993,INC0029231,Resolved,True,0,0,8,True,Caller 2321,Opened by 17,13/5/2016 11:46,?,?,Updated by 44,13/5/2016 12:15,Phone,Location 111,Category 32,Subcategory 9,?,2 - Medium,2 - Medium,3 - Moderate,Group 70,Resolver 13,False,True,Do Not Notify,code 7,Resolved by 11,13/5/2016 12:15,18/5/2016 13:07
119994,INC0029231,Closed,False,0,0,9,True,Caller 2321,Opened by 17,13/5/2016 11:46,?,?,Updated by 908,18/5/2016 13:07,Phone,Location 111,Category 32,Subcategory 9,?,2 - Medium,2 - Medium,3 - Moderate,Group 70,Resolver 13,False,True,Do Not Notify,code 7,Resolved by 11,13/5/2016 12:15,18/5/2016 13:07
119995,INC0029232,Resolved,True,0,0,0,True,Caller 382,Opened by 108,13/5/2016 11:48,?,?,Updated by 908,13/5/2016 11:48,Phone,Location 143,Category 53,Subcategory 174,?,2 - Medium,2 - Medium,3 - Moderate,Group 70,Resolver 56,False,False,Do Not Notify,code 6,Resolved by 52,?,18/5/2016 12:07
119996,INC0029232,Closed,False,0,0,1,True,Caller 382,Opened by 108,13/5/2016 11:48,?,?,Updated by 908,18/5/2016 12:07,Phone,Location 143,Category 53,Subcategory 174,?,2 - Medium,2 - Medium,3 - Moderate,Group 70,Resolver 56,False,False,Do Not Notify,code 6,Resolved by 52,?,18/5/2016 12:07


In [177]:
#seeing that there are multiple rows per incident, we can see that the first 4 rows all have the same number, but show progression in incident_state
#df.head()
print(f"Number of rows in 'resolved_at' containing a missing value = {(df['resolved_at'] == '?').sum()}")
#datetime_obj = pd.to_datetime(df['resolved_at'], format="%d/%m/%Y %H:%M")
#print(datetime_obj)
print(df.shape)
df['resolved_at'].sample(n=20) # shown to be day/month/year/time


#df['closed_at'].sample(n=10) # shown to be day/month/year

Number of rows in 'resolved_at' containing a missing value = 2861
(119998, 36)


112260    14/6/2016 08:26
52233     30/3/2016 17:03
105550                  ?
3290       2/3/2016 11:06
82399     18/4/2016 12:12
55862      5/4/2016 12:36
80335     14/4/2016 14:56
62106     11/4/2016 08:16
14728     17/3/2016 04:34
47342     31/3/2016 10:40
78429     28/4/2016 09:33
18385     21/3/2016 12:58
874       22/3/2016 12:57
51840     28/3/2016 11:15
7361       9/3/2016 13:43
49012     15/4/2016 19:03
22925     21/3/2016 08:42
31340     17/3/2016 14:31
108534     5/5/2016 10:47
52579     28/3/2016 16:14
Name: resolved_at, dtype: object

A subset of incidents (≈2.3%) were closed without a recorded resolution timestamp. Inspection of event indices confirms that resolved_at is missing for all rows of those incidents, indicating that resolution time was never entered into the system and cannot be recovered. These incidents are retained and flagged rather than imputed or dropped.

In [178]:

df['reassignment_count'].iloc[30:60]

df.iloc[56:58]

,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,sys_created_by,sys_created_at,sys_updated_by,sys_updated_at,contact_type,location,category,subcategory,u_symptom,cmdb_ci,impact,urgency,priority,assignment_group,assigned_to,knowledge,u_priority_confirmation,notify,problem_id,rfc,vendor,caused_by,closed_code,resolved_by,resolved_at,closed_at
56,INC0000065,New,True,4,0,8,True,Caller 5323,Opened by 131,29/2/2016 07:38,Created by 62,29/2/2016 07:46,Updated by 750,29/2/2016 11:17,Phone,Location 108,Category 45,Subcategory 229,Symptom 580,?,2 - Medium,2 - Medium,3 - Moderate,Group 12,Resolver 216,True,False,Do Not Notify,?,?,?,?,code 1,Resolved by 197,2/3/2016 15:21,7/3/2016 16:00
57,INC0000065,New,True,5,0,9,True,Caller 5323,Opened by 131,29/2/2016 07:38,Created by 62,29/2/2016 07:46,Updated by 798,29/2/2016 14:51,Phone,Location 108,Category 45,Subcategory 229,Symptom 580,?,2 - Medium,2 - Medium,3 - Moderate,Group 15,Resolver 216,True,False,Do Not Notify,?,?,?,?,code 1,Resolved by 197,2/3/2016 15:21,7/3/2016 16:00


In [179]:
# It's time to aggregate the dataset now from event -> incident per row

#we create a new boolean feature to show whether the data is missing the resolved_at column, as dropping it would hurt data integrity
# as stated earlier, imputation here is impossible as there isn't enough data to make realistic predictions on what the resolved_at time would be
df['has_resolution_time'] = df['resolved_at'] == '?'


df['closed_without_resolution'] = (df['resolved_at'] =='?') & (df['closed_at'] != '?') 

time_cols = ["opened_at", "resolved_at", "closed_at"]

for col in time_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")
df["has_resolution_time"] = df["resolved_at"].notna().astype(int)

df["closed_without_resolution"] = (
    df["resolved_at"].isna() & df["closed_at"].notna()
).astype(int)
df["resolution_time_hours"] = (
    df["resolved_at"] - df["opened_at"]
).dt.total_seconds() / 3600

df["closure_time_hours"] = (
    df["closed_at"] - df["opened_at"]
).dt.total_seconds() / 3600

df["post_resolution_delay_hours"] = (
    df["closed_at"] - df["resolved_at"]
).dt.total_seconds() / 3600

df['opened_at'] = pd.to_datetime(df['opened_at'], format='%d/%m/%Y %H:%M')
df['opened_dayofweek'] = df['opened_at'].dt.dayofweek  # 0=Monday, 6=Sunday
df['opened_hour'] = df['opened_at'].dt.hour  # Hour of the day (0-23)
df['opened_is_weekend'] = df['opened_dayofweek'].isin([5, 6]).astype(int)  # 1 if Saturday/Sunday, else 0




df.head()


C:\Users\Vishnu Ravi\AppData\Local\Temp\ipykernel_51768\1500197834.py:13: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors="coerce")
C:\Users\Vishnu Ravi\AppData\Local\Temp\ipykernel_51768\1500197834.py:13: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors="coerce")


,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,sys_created_by,sys_created_at,sys_updated_by,sys_updated_at,contact_type,location,category,subcategory,u_symptom,cmdb_ci,impact,urgency,priority,assignment_group,assigned_to,knowledge,u_priority_confirmation,notify,problem_id,rfc,vendor,caused_by,closed_code,resolved_by,resolved_at,closed_at,has_resolution_time,closed_without_resolution,resolution_time_hours,closure_time_hours,post_resolution_delay_hours,opened_dayofweek,opened_hour,opened_is_weekend
0,INC0000045,New,True,0,0,0,True,Caller 2403,Opened by 8,2016-02-29 01:16:00,Created by 6,29/2/2016 01:23,Updated by 21,29/2/2016 01:23,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,?,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,2016-02-29 11:29:00,2016-05-03 12:00:00,1,0,10.216667,1546.733333,1536.516667,0,1,0
1,INC0000045,Resolved,True,0,0,2,True,Caller 2403,Opened by 8,2016-02-29 01:16:00,Created by 6,29/2/2016 01:23,Updated by 642,29/2/2016 08:53,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,?,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,2016-02-29 11:29:00,2016-05-03 12:00:00,1,0,10.216667,1546.733333,1536.516667,0,1,0
2,INC0000045,Resolved,True,0,0,3,True,Caller 2403,Opened by 8,2016-02-29 01:16:00,Created by 6,29/2/2016 01:23,Updated by 804,29/2/2016 11:29,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,?,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,2016-02-29 11:29:00,2016-05-03 12:00:00,1,0,10.216667,1546.733333,1536.516667,0,1,0
3,INC0000045,Closed,False,0,0,4,True,Caller 2403,Opened by 8,2016-02-29 01:16:00,Created by 6,29/2/2016 01:23,Updated by 908,5/3/2016 12:00,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,?,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,2016-02-29 11:29:00,2016-05-03 12:00:00,1,0,10.216667,1546.733333,1536.516667,0,1,0
4,INC0000047,New,True,0,0,0,True,Caller 2403,Opened by 397,2016-02-29 04:40:00,Created by 171,29/2/2016 04:57,Updated by 746,29/2/2016 04:57,Phone,Location 165,Category 40,Subcategory 215,Symptom 471,?,2 - Medium,2 - Medium,3 - Moderate,Group 70,Resolver 89,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 81,2016-03-01 09:52:00,2016-06-03 10:00:00,1,0,29.200000,2285.333333,2256.133333,0,4,0


In [180]:
df = df.replace("?", np.nan)

datetime_cols = [
    "opened_at",
    "resolved_at",
    "closed_at",
    "sys_created_at",
    "sys_updated_at"
]

for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")


df = df.sort_values(["number", "sys_updated_at"])
incident_df = (
    df
    .groupby("number")
    .agg(opened_at=("opened_at", "min"),
        resolved_at=("resolved_at", "max"),
        closed_at=("closed_at", "max"),reassignment_count=("reassignment_count", "max"),
        reopen_count=("reopen_count", "max"),
        sys_mod_count=("sys_mod_count", "max"),initial_priority=("priority", "first"),
        final_priority=("priority", "last"),
        max_priority=("priority", "max"),assignment_group=("assignment_group", "last"),
        assigned_to=("assigned_to", "last"),made_sla=("made_sla", "last"),
        knowledge=("knowledge", "max"),)
    .reset_index()
)







C:\Users\Vishnu Ravi\AppData\Local\Temp\ipykernel_51768\53548520.py:12: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors="coerce")
C:\Users\Vishnu Ravi\AppData\Local\Temp\ipykernel_51768\53548520.py:12: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors="coerce")


In [181]:
state_changes = (
    df
    .groupby("number")["incident_state"]
    .apply(lambda x: x.ne(x.shift()).sum() - 1)
)

incident_df["state_change_count"] = incident_df["number"].map(state_changes)

incident_df["has_resolution_timestamp"] = (
    incident_df["resolved_at"].notna().astype(int)
)

incident_df["closed_without_resolution"] = (
    incident_df["resolved_at"].isna().astype(int)
)
incident_df["resolution_time_hours"] = (
    incident_df["resolved_at"] - incident_df["opened_at"]
).dt.total_seconds() / 3600

incident_df["closure_time_hours"] = (
    incident_df["closed_at"] - incident_df["opened_at"]
).dt.total_seconds() / 3600

incident_df["post_resolution_delay_hours"] = (
    incident_df["closed_at"] - incident_df["resolved_at"]
).dt.total_seconds() / 3600



incident_df["opened_dayofweek"] = incident_df["opened_at"].dt.dayofweek
incident_df["opened_hour"] = incident_df["opened_at"].dt.hour
incident_df["opened_is_weekend"] = (
    incident_df["opened_dayofweek"].isin([5, 6]).astype(int)
)


incident_df["priority_escalated"] = (
    incident_df["max_priority"] > incident_df["initial_priority"]
).astype(int)

